# 02 Energy, Services and Supercharging

Energy storage and recurring services can matter a lot if they compound
with higher margins than auto. This notebook separates those cash-flow
engines from FSD/robotaxi option value.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from valuation import (
    MarketPremiumInputs,
    OptionScenario,
    SegmentValuation,
    SotpInputs,
    build_sensitivity_grid,
    implied_margin_for_enterprise_value,
    implied_revenue_for_enterprise_value,
    market_premium_value,
    probability_weighted_option_value,
    sotp_equity_value,
)

DATA_DIR = next(
    candidate / "apps/notebooks/studies/tesla_valuation/data"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "apps/notebooks/studies/tesla_valuation/data").exists()
)
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
assumptions = pd.read_csv(DATA_DIR / "segment_assumptions.csv")
cases = ["bear", "base", "bull"]
segment_tables = {}
for segment in ["Energy", "Services"]:
    segment_tables[segment] = (
        assumptions[assumptions["segment"].eq(segment)]
        .pivot_table(index="case", columns="metric", values="value", aggfunc="first")
        .reindex(cases)
    )
segment_tables["Energy"]

In [ ]:
energy = segment_tables["Energy"].assign(
    ebitda_usd_bn=lambda df: df["2030 revenue"] * df["EBITDA margin"],
    ev_revenue_multiple=[2.0, 4.0, 6.0],
    ev_ebitda_multiple=[15.0, 22.0, 30.0],
)
energy["selected_value_usd_bn"] = (
    energy["2030 revenue"] * energy["ev_revenue_multiple"]
    + energy["ebitda_usd_bn"] * energy["ev_ebitda_multiple"]
) / 2
energy

In [ ]:
services = segment_tables["Services"].assign(
    ebitda_usd_bn=lambda df: df["2030 revenue"] * df["EBITDA margin"],
    ev_revenue_multiple=[1.5, 2.5, 4.0],
    ev_ebitda_multiple=[12.0, 18.0, 25.0],
)
services["selected_value_usd_bn"] = (
    services["2030 revenue"] * services["ev_revenue_multiple"]
    + services["ebitda_usd_bn"] * services["ev_ebitda_multiple"]
) / 2
services

In [ ]:
storage_grid = build_sensitivity_grid(
    row_values=[100, 200, 300, 400, 500],
    column_values=[0.10, 0.15, 0.20, 0.25, 0.30],
    formula=lambda gwh, gross_margin: gwh * 0.25 * gross_margin,
    row_name="Storage deployments (GWh)",
    column_name="Storage gross margin",
)
storage_grid